# Soft Prompt Tuning - Condition 2 (Lightweight)

| Property | Value |
|---|---|
| Training data | CDA-augmented (`train_augmented.csv`) |
| Trainable params | 20 soft prompt vectors + classifier head (~16,900) |
| Backbone | Frozen |
| Fairness loss | None |
| Class weights | From `data/class_weights.json` (shared) |


## 1. Install packages

In [ ]:
# Install PyTorch with CUDA 12.1 support for RTX 4060 Ti
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121 --quiet
%pip install transformers==4.44.0 datasets==2.20.0 scikit-learn pandas numpy==1.26.4 codecarbon scipy --quiet

## 2. Imports

In [ ]:
import os
import json
import time
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.special import softmax

from datasets import Dataset, DatasetDict
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
)

from transformers import (
    AutoTokenizer,
    AutoModel,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed,
)
from transformers.modeling_outputs import SequenceClassifierOutput

from codecarbon import EmissionsTracker

## 3. Configuration

In [ ]:
# Model
MODEL_NAME = "bert-base-uncased"
#MODEL_NAME = "distilbert-base-uncased"
#MODEL_NAME = "roberta-base"

# Soft prompt
N_PROMPT_TOKENS = 20   # learnable vectors prepended to every input

# Run mode
RUN_MODE = "STANDARD"   # "DEBUG" | "STANDARD"
if RUN_MODE == "DEBUG":
    MAX_LENGTH    = 64
    EPOCHS        = 1
    TRAIN_BATCH   = 4
    EVAL_BATCH    = 4
    SAVE_STRATEGY = "no"
    LOAD_BEST     = False
elif RUN_MODE == "STANDARD":
    MAX_LENGTH    = 128
    EPOCHS        = 3
    TRAIN_BATCH   = 16
    EVAL_BATCH    = 32
    SAVE_STRATEGY = "no"
    LOAD_BEST     = False
# Paths
BASE_DIR    = r"C:\Users\scanc\Desktop\v3"
DATA_DIR    = os.path.join(BASE_DIR, "data")
RESULTS_DIR = os.path.join(BASE_DIR, "baseline_results")

os.makedirs(RESULTS_DIR, exist_ok=True)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

safe_model_name = MODEL_NAME.replace("/", "_").replace("-", "_") + "final_test"

print(f"Model            : {MODEL_NAME}")
print(f"N prompt tokens  : {N_PROMPT_TOKENS}")
print(f"Run mode         : {RUN_MODE}")
print(f"Text max length  : {MAX_LENGTH} tokens (same as baseline)")
print(f"Prompt tokens    : {N_PROMPT_TOKENS} prepended")
print(f"Total sequence   : {MAX_LENGTH + N_PROMPT_TOKENS} tokens")
print(f"Data dir         : {DATA_DIR}")
print(f"Results dir      : {RESULTS_DIR}")

## 4. Check device

In [ ]:
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    device = "cuda"
    print("GPU :", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory/1024**3,1), "GB")
elif torch.backends.mps.is_available():
    device = "mps"
    print("Using Apple MPS")
else:
    device = "cpu"
    print("No GPU - CPU only")
print("Active device:", device)

## 5. Load shared splits and class weights

In [ ]:
# Load CDA-augmented training set
train_df = pd.read_csv(os.path.join(DATA_DIR, "train_augmented.csv"))
val_df   = pd.read_csv(os.path.join(DATA_DIR, "val.csv"))
test_df  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

with open(os.path.join(DATA_DIR, "class_weights.json")) as f:
    cw = json.load(f)
class_weights_dict = {int(k): v for k, v in cw.items()}

print(f"Train (augmented) : {len(train_df):,}")
print(f"Val               : {len(val_df):,}")
print(f"Test              : {len(test_df):,}")
print(f"Class weights     : {class_weights_dict}")

if RUN_MODE == "DEBUG":
    train_df = train_df.sample(n=min(2000, len(train_df)), random_state=SEED).reset_index(drop=True)
    val_df   = val_df.sample(n=min(200,  len(val_df)),   random_state=SEED).reset_index(drop=True)
    test_df  = test_df.sample(n=min(400,  len(test_df)),  random_state=SEED).reset_index(drop=True)
    print(f"\nDEBUG: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")

## 6. Soft Prompt Classifier

20 learnable embedding vectors are prepended to every input at the embedding level.
Only the prompt vectors and classifier head are trained.

Trainable parameters:
- Prompt embeddings: 20 × 768 = 15,360
- Classifier head: 768 × 2 + 2 = 1,538
- Total: ~16,898

In [ ]:
class SoftPromptClassifier(nn.Module):

    def __init__(self, model_name, n_prompt_tokens, class_weights_dict, num_labels=2):
        super().__init__()
        self.n_prompt_tokens = n_prompt_tokens
        self.is_distilbert   = "distilbert" in model_name.lower()
        self.is_roberta      = "roberta" in model_name.lower()

        # Load and freeze backbone
        self.backbone = AutoModel.from_pretrained(model_name)
        for param in self.backbone.parameters():
            param.requires_grad = False

        self.hidden_size = self.backbone.config.hidden_size

        # Initialise prompt from random vocab embeddings (Lester et al. 2021)
        with torch.no_grad():
            vocab_embeds   = self.backbone.embeddings.word_embeddings.weight
            random_indices = torch.randint(0, vocab_embeds.size(0), (n_prompt_tokens,))
            init_embeds    = vocab_embeds[random_indices].clone()
        self.prompt_embeddings = nn.Parameter(init_embeds)

        # Classification head
        dropout_p       = getattr(self.backbone.config, "hidden_dropout_prob",
                          getattr(self.backbone.config, "dropout", 0.1))
        self.dropout    = nn.Dropout(dropout_p)
        self.classifier = nn.Linear(self.hidden_size, num_labels)

        # Store class weights as a buffer (moves to correct device automatically)
        self.register_buffer(
            "class_weights",
            torch.tensor(
                [class_weights_dict[0], class_weights_dict[1]],
                dtype=torch.float32,
            ),
        )

        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total     = sum(p.numel() for p in self.parameters())
        print(f"  Trainable : {trainable:,}")
        print(f"  Total     : {total:,}")
        print(f"  Frozen    : {total - trainable:,}")

    def forward(self, input_ids, attention_mask, labels=None, token_type_ids=None):
        batch_size = input_ids.size(0)

        # Get token embeddings
        token_embeds = self.backbone.embeddings.word_embeddings(input_ids)

        # Expand and prepend prompt
        prompt        = self.prompt_embeddings.unsqueeze(0).expand(batch_size, -1, -1)
        inputs_embeds = torch.cat([prompt, token_embeds], dim=1)

        # Extend attention mask with 1s for prompt positions
        prompt_mask   = torch.ones(batch_size, self.n_prompt_tokens,
                                   device=attention_mask.device, dtype=attention_mask.dtype)
        extended_mask = torch.cat([prompt_mask, attention_mask], dim=1)

        # Forward through backbone
        if self.is_distilbert:
            outputs = self.backbone(
                inputs_embeds  = inputs_embeds,
                attention_mask = extended_mask,
            )
        else:
            if token_type_ids is not None:
                prompt_type   = torch.zeros(batch_size, self.n_prompt_tokens,
                                            device=token_type_ids.device,
                                            dtype=token_type_ids.dtype)
                extended_type = torch.cat([prompt_type, token_type_ids], dim=1)
            else:
                extended_type = None
            outputs = self.backbone(
                inputs_embeds  = inputs_embeds,
                attention_mask = extended_mask,
                token_type_ids = extended_type,
            )

        # Use [CLS] token representation.
        # Prompts are prepended so original [CLS] is now at index n_prompt_tokens.
        pooled = outputs.last_hidden_state[:, self.n_prompt_tokens, :]

        logits = self.classifier(self.dropout(pooled))

        loss = None
        if labels is not None:
            weights = self.class_weights.to(device=logits.device, dtype=logits.dtype)
            loss    = nn.functional.cross_entropy(
                logits, labels.to(logits.device), weight=weights
            )

        return SequenceClassifierOutput(loss=loss, logits=logits)


## 7. Tokenize

Effective max length = `MAX_LENGTH - N_PROMPT_TOKENS` so the total
sequence (prompt + text) never exceeds the model's positional limit.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Text max length is NOT reduced by prompt tokens.
# Baseline sees: [CLS] + 128-token comment window
# Prompt tuning sees: 20 prompt tokens + [CLS] + 128-token comment window
# Total = 148 tokens - well within BERT's 512 limit.
# This keeps the text input comparable to the baseline.
EFFECTIVE_MAX          = MAX_LENGTH
TOTAL_SEQUENCE_LENGTH  = MAX_LENGTH + N_PROMPT_TOKENS

print(f"Text max length        : {EFFECTIVE_MAX} tokens")
print(f"Prompt tokens          : {N_PROMPT_TOKENS}")
print(f"Total sequence length  : {TOTAL_SEQUENCE_LENGTH} tokens")

if TOTAL_SEQUENCE_LENGTH > tokenizer.model_max_length:
    raise ValueError(
        f"Total sequence length {TOTAL_SEQUENCE_LENGTH} exceeds "
        f"model limit {tokenizer.model_max_length}."
    )

def preprocess(examples):
    return tokenizer(
        examples["comment_text"],
        truncation=True,
        max_length=EFFECTIVE_MAX,
    )

train_hf = Dataset.from_pandas(train_df[["comment_text","label"]].reset_index(drop=True))
val_hf   = Dataset.from_pandas(val_df[["comment_text","label"]].reset_index(drop=True))
test_hf  = Dataset.from_pandas(test_df[["comment_text","label"]].reset_index(drop=True))

hf_ds    = DatasetDict({"train": train_hf, "validation": val_hf, "test": test_hf})
tokenized = hf_ds.map(preprocess, batched=True)

keep = ["input_ids", "attention_mask", "token_type_ids", "label"]
for split in tokenized:
    to_remove = [c for c in tokenized[split].column_names if c not in keep]
    tokenized[split] = tokenized[split].remove_columns(to_remove)
tokenized.set_format("torch")
print(tokenized)

## 8. Instantiate model

In [ ]:
print(f"Building SoftPromptClassifier for {MODEL_NAME}...")
model = SoftPromptClassifier(
    model_name         = MODEL_NAME,
    n_prompt_tokens    = N_PROMPT_TOKENS,
    class_weights_dict = class_weights_dict,
    num_labels         = 2,
)

## 9. Metrics

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    logits = np.array(logits, dtype=np.float32)

    predictions   = np.argmax(logits, axis=1)
    probabilities = softmax(logits, axis=1)[:, 1]

    accuracy                = accuracy_score(labels, predictions)
    p_bin, r_bin, f1_bin, _ = precision_recall_fscore_support(
        labels, predictions, average="binary",   zero_division=0)
    _, _, f1_w, _           = precision_recall_fscore_support(
        labels, predictions, average="weighted", zero_division=0)
    try:
        auc = roc_auc_score(labels, probabilities)
    except ValueError:
        auc = float("nan")

    return {
        "accuracy"    : round(accuracy, 4),
        "precision"   : round(p_bin,    4),
        "recall"      : round(r_bin,    4),
        "f1_binary"   : round(f1_bin,   4),
        "f1_weighted" : round(f1_w,     4),
        "auc_roc"     : round(auc,      4),
    }

## 10. Output folders

In [ ]:
output_dir  = os.path.join(RESULTS_DIR, f"{safe_model_name}_prompt_tuning_model")
logging_dir = os.path.join(RESULTS_DIR, f"{safe_model_name}_prompt_tuning_logs")
os.makedirs(output_dir,  exist_ok=True)
os.makedirs(logging_dir, exist_ok=True)
print("Output dir:", output_dir)

## 11. Training arguments

Learning rate 1e-3 - higher than baseline because only ~16,900 parameters
are being updated. Standard 2e-5 would move the prompt vectors too slowly.

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir    = output_dir,
    eval_strategy = "epoch",
    save_strategy = SAVE_STRATEGY,

    learning_rate               = 1e-3,
    per_device_train_batch_size = TRAIN_BATCH,
    per_device_eval_batch_size  = EVAL_BATCH,
    num_train_epochs            = EPOCHS,
    weight_decay                = 0.01,

    logging_dir   = logging_dir,
    logging_steps = 10,

    load_best_model_at_end = LOAD_BEST,
    metric_for_best_model  = "f1_binary" if LOAD_BEST else None,
    greater_is_better      = True,

    report_to             = "none",
    seed                  = SEED,
    remove_unused_columns = False,
)

## 12. Train with CodeCarbon tracking

In [ ]:
tracker = EmissionsTracker(
    project_name = f"{safe_model_name}_prompt_tuning",
    output_dir   = RESULTS_DIR,
    output_file  = f"{safe_model_name}_prompt_tuning_emissions.csv",
    log_level    = "warning",
    save_to_file = True,
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = tokenized["train"],
    eval_dataset    = tokenized["validation"],
    compute_metrics = compute_metrics,
    data_collator   = data_collator,
)

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

tracker.start()
wall_start = time.time()

trainer.train()

wall_end        = time.time()
emissions_kg    = tracker.stop()
runtime_seconds = wall_end - wall_start

print(f"\nTraining complete.")
print(f"  Wall-clock time : {runtime_seconds:.1f} s  ({runtime_seconds/60:.2f} min)")
print(f"  CO2 emitted     : {emissions_kg:.6f} kg CO2eq")

## 13. Final evaluation on test set

In [ ]:
results = trainer.evaluate(eval_dataset=tokenized["test"])
print("\nTest set results:")
for k, v in results.items():
    print(f"  {k:<30} {v}")

## 14. Save performance results

In [ ]:
results_df = pd.DataFrame([results])
results_df["model"]           = MODEL_NAME
results_df["method"]          = "prompt_tuning"
results_df["run_mode"]        = RUN_MODE
results_df["train_size"]      = len(train_df)
results_df["val_size"]        = len(val_df)
results_df["test_size"]       = len(test_df)
results_df["cda"]             = True
results_df["fairness_loss"]   = False
results_df["n_prompt_tokens"] = N_PROMPT_TOKENS
results_df["max_length"]      = MAX_LENGTH
results_df["epochs"]          = EPOCHS
results_df["runtime_seconds"] = runtime_seconds
results_df["runtime_minutes"] = runtime_seconds / 60

path = os.path.join(RESULTS_DIR, f"{safe_model_name}_prompt_tuning_results.csv")
results_df.to_csv(path, index=False)
print("Saved:", path)
results_df

## 15. Save predictions

In [ ]:
pred_output   = trainer.predict(tokenized["test"])
logits        = pred_output.predictions
if isinstance(logits, tuple):
    logits = logits[0]
logits        = np.array(logits, dtype=np.float32)
probabilities = softmax(logits, axis=1)[:, 1]
predictions   = (probabilities >= 0.5).astype(int)
true_labels   = pred_output.label_ids

prediction_df = test_df.reset_index(drop=True).copy()
prediction_df["true_label"]     = true_labels
prediction_df["toxicity_score"] = probabilities
prediction_df["prediction"]     = predictions
prediction_df["model"]          = MODEL_NAME
prediction_df["method"]         = "prompt_tuning"

path = os.path.join(RESULTS_DIR, f"{safe_model_name}_prompt_tuning_predictions.csv")
prediction_df.to_csv(path, index=False)
print("Saved:", path)
print(f"Rows : {len(prediction_df):,}")
prediction_df.head()

## 16. Save sustainability metrics

In [ ]:
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
peak_gpu  = torch.cuda.max_memory_allocated()/1024**3 if torch.cuda.is_available() else None

try:
    cc_df      = pd.read_csv(os.path.join(RESULTS_DIR, f"{safe_model_name}_prompt_tuning_emissions.csv"))
    energy_kwh = cc_df["energy_consumed"].iloc[-1]
except Exception:
    energy_kwh = None

sustainability_df = pd.DataFrame([{
    "model"                : MODEL_NAME,
    "method"               : "prompt_tuning",
    "cda"                  : True,
    "fairness_loss"        : False,
    "n_prompt_tokens"      : N_PROMPT_TOKENS,
    "run_mode"             : RUN_MODE,
    "train_size"           : len(train_df),
    "val_size"             : len(val_df),
    "test_size"            : len(test_df),
    "runtime_seconds"      : round(runtime_seconds, 2),
    "runtime_minutes"      : round(runtime_seconds / 60, 4),
    "energy_kwh"           : energy_kwh,
    "emissions_kg_co2eq"   : round(emissions_kg, 8) if emissions_kg else None,
    "trainable_parameters" : trainable,
    "total_parameters"     : total,
    "trainable_pct"        : round(100 * trainable / total, 6),
    "peak_gpu_memory_gb"   : round(peak_gpu, 4) if peak_gpu else None,
    "max_length"             : MAX_LENGTH,
    "text_max_length"        : EFFECTIVE_MAX,
    "total_sequence_length"  : TOTAL_SEQUENCE_LENGTH,
    "epochs"                 : EPOCHS,
}])

path = os.path.join(RESULTS_DIR, f"{safe_model_name}_prompt_tuning_sustainability.csv")
sustainability_df.to_csv(path, index=False)
print("Saved:", path)
sustainability_df